# Prompt Chaining Workflow Example

Start -> Topic -> Outliune -> Content -> End

In [ ]:
# !pip install langchain langgraph dotenv

In [ ]:
# load environment variables from .env file
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List

In [ ]:
# Define state

class AgentState(TypedDict):
    title: str
    outline: str
    content: str

In [ ]:
# Create a OpenAI model instance
from langchain_openai import ChatOpenAI

# model = ChatOpenAI()


model = ChatOpenAI(    
    base_url="http://localhost:12434/engines/v1",
    api_key="docker", 
    temperature=0, 
    model = "ai/smollm2:360M-Q4_K_M")

In [ ]:
# create outline
def create_outline(state: AgentState) -> AgentState:
    """Create an outline for the article based on the title."""
    # Placeholder implementation - replace with actual outline creation logic
    # return {"outline": f"Outline for {state['title']}"}

    # Fetch title from state
    title = state["title"]

    # Generate outline based on title using LLM (placeholder logic)
    outline = f"Generate a detailed outline on the topic - {title}"
    outline = model.invoke(outline).content

    state["outline"] = outline
    return state


# Cretae content
def create_content(state: AgentState) -> AgentState:
    """Create content for the article based on the outline."""
    # Placeholder implementation - replace with actual content creation logic
    # return {"content": f"Content based on {state['outline']}"}
    
    # Fetch title and outline from state
    title = state["title"]
    outline = state["outline"]

    # Generate content based on outline using LLM (placeholder logic)
    content = f"Generate content for the article topic - {title}\n\nbased on this outline: {outline}"
    content = model.invoke(content).content

    state["content"] = content
    return state

In [ ]:
# Build graph

graph = StateGraph(AgentState)

# Add nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_content', create_content)

# add edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_content')
graph.add_edge('create_content', END)

# compile graph
workflow = graph.compile()


In [ ]:

# visualize graph
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())


In [ ]:

# execute graph
initial_state = AgentState({
    'title': "State of AI in India",
    'outline': "",
    'content': ""
})

final_state = workflow.invoke(initial_state)
print("Title:", final_state["title"])
print("\n\nOutline:", final_state["outline"])
print("\n\nContent:", final_state["content"])